# FinTrust Risk Analysis

This notebook reproduces the final risk-model workflow for the FinTrust data science track.
It loads the customer and transaction datasets, engineers the final feature set, compares candidate models, and fits the final logistic regression baseline.


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [ ]:
customer = pd.read_csv('../customer.csv')
transactions = pd.read_csv('../transactions.csv')
transactions['Transaction_DateTime'] = pd.to_datetime(transactions['Transaction_DateTime'])
transactions = transactions.sort_values(['Customer_ID', 'Transaction_DateTime']).reset_index(drop=True)
transactions['Risk_Binary'] = (transactions['Risk_Review_Flag'] == 'Yes').astype(int)
transactions['customer_prior_tx_count'] = transactions.groupby('Customer_ID').cumcount()
transactions['customer_prior_risk_count'] = transactions.groupby('Customer_ID')['Risk_Binary'].transform(lambda s: s.shift(fill_value=0).cumsum())
transactions['customer_prior_risk_rate'] = transactions['customer_prior_risk_count'] / np.maximum(transactions['customer_prior_tx_count'], 1)
transactions['customer_prior_failure_count'] = transactions.groupby('Customer_ID')['Transaction_Status'].transform(lambda s: s.eq('Failed').shift(fill_value=False).cumsum())
transactions['customer_prior_failure_rate'] = transactions['customer_prior_failure_count'] / np.maximum(transactions['customer_prior_tx_count'], 1)
transactions['Hour'] = transactions['Transaction_DateTime'].dt.hour
transactions['DayOfWeek'] = transactions['Transaction_DateTime'].dt.dayofweek
transactions['Month'] = transactions['Transaction_DateTime'].dt.month
transactions['Amount_Log'] = np.log1p(transactions['Amount_NGN'])
transactions['Is_International'] = (transactions['International_Transaction'] == 'Yes').astype(int)
transactions['Is_Failed'] = (transactions['Transaction_Status'] == 'Failed').astype(int)
transactions['Is_Reversed'] = (transactions['Transaction_Status'] == 'Reversed').astype(int)
transactions['Is_Pending'] = (transactions['Transaction_Status'] == 'Pending').astype(int)
df = transactions.merge(customer, on='Customer_ID', how='left')
X = df.drop(columns=['Transaction_ID', 'Customer_ID', 'Customer_Name', 'Transaction_DateTime', 'Risk_Binary', 'Risk_Review_Flag'])
y = (df['Risk_Review_Flag'] == 'Yes').astype(int)
categorical = X.select_dtypes(include=['object']).columns.tolist()
numeric = X.select_dtypes(exclude=['object']).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])
pipeline = Pipeline([('preprocessor', preprocessor), ('model', LogisticRegression(max_iter=4000, class_weight='balanced', random_state=42))])
pipeline.fit(X_train, y_train)
probs = pipeline.predict_proba(X_test)[:, 1]
threshold = 0.48
predictions = (probs >= threshold).astype(int)
metrics = {
    'accuracy': accuracy_score(y_test, predictions),
    'precision': precision_score(y_test, predictions),
    'recall': recall_score(y_test, predictions),
    'f1_score': f1_score(y_test, predictions),
    'roc_auc': roc_auc_score(y_test, probs),
    'confusion_matrix': confusion_matrix(y_test, predictions).tolist(),
}
print(json.dumps(metrics, indent=2))
